Imagine you're working as a Data Engineer for an e-commerce company.

The source system sends you an orders dataset every day.

Unfortunately, the source data is dirty.

Create DataFrame

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Day 2").getOrCreate()
print(spark)

In [0]:
data = [
    ("O101", " C001 ", "Laptop", "1500.50", 2, "2026-08-01"),
    ("O102", "C002", "Mobile", "800", None, "2026-08-01"),
    ("O103", " C003", "Tablet", "-500", 1, "2026-08-02"),
    ("O104", "C004 ", "Laptop", "1200.75", 1, "invalid_date"),
    ("O105", "C005", "Mobile", "900", 2, "2026-08-03"),
    ("O105", "C005", "Mobile", "900", 2, "2026-08-03"),
    ("O106", None, "Tablet", "700", 1, "2026-08-03"),
]

columns = [
    "order_id",
    "customer_id",
    "product",
    "amount",
    "quantity",
    "order_date"
]

df=spark.createDataFrame(data, columns)
df.show()
df.printSchema()

##### The business team tells you:
#####
###### We cannot directly use this data for reporting. Clean the dataset and create a reliable orders table.

Problem 1 — Customer IDs

Some `customer_id `values contain unnecessary spaces.

In [0]:
from pyspark.sql.functions import *

In [0]:
df=df.withColumn('customer_id',trim(col('customer_id')))
df.show()

Problem 2 — Amount

Currently:

`amount
- 1500.50
- 800
- -500`

The schema will show that amount is probably a string.

Requirement

Convert amount into a proper numeric datatype.

Question: Which datatype would you choose for a money column?

In [0]:
df=df.withColumn('amount',col('amount').cast('float'))
df.show()
df.printSchema()

Problem 3 — Missing Quantity

One order has:

`quantity = NULL`

Business rule:

If quantity is missing, assume quantity = 1.

Requirement

Replace NULL quantity with 1.

Write the code.

In [0]:
from pyspark.sql.functions import when, col, lit
df=df.withColumn('quantity', when(col('quantity').isNull(), lit(1)).otherwise(col('quantity')))
df.show()

Problem 4 — Invalid Amount

There is an order:

amount = -500

Business rule:

An order amount cannot be negative.

We don't want to delete the order yet.

Instead, create a new column:

amount_status

In [0]:
df=df.withColumn('amount_status',when(col('amount')<0,lit('Invalid') ).otherwise(lit('Valid')))

df.show()

In [0]:
df.show()

Problem 5 — Order Date

One record contains:

invalid_date

The business wants order_date as a proper date.

Requirement

Convert order_date from string → date.

But be careful:

Invalid dates should become NULL rather than crashing the pipeline.

In [0]:
from pyspark.sql.functions import try_to_date

In [0]:
df=df.withColumn('order_date',try_to_date(col('order_date'),'yyyy-MM-dd'))
df.show()

Problem 6 — Duplicate Orders

You notice:

O105
O105

The same order appears twice.

Business rule:

order_id should uniquely identify an order.

Requirement

Remove duplicate orders.

In [0]:
df=df.dropDuplicates()
df.show()
df.printSchema()

Problem 7 — Missing Customer

One order has:

customer_id = NULL

Business rule:

Orders without a customer cannot be used for customer-level reporting.

Remove those records.

In [0]:
df=df.dropna(subset=['customer_id'])
df.show()

Final Business Requirement

After cleaning, create a new column:

order_value_category

Rules:

amount >= 1000       → "HIGH"
amount >= 500        → "MEDIUM"
amount < 500         → "LOW"

For example:

amount	category
1500.50	HIGH
800	MEDIUM
700	MEDIUM
1200.75	HIGH

But remember our -500 record is invalid.

So think about whether you should classify it before or after handling invalid amounts.

In [0]:
df=df.withColumn('order_value_category',when(col('amount_status')=='Valid',\
    when(col('amount')>=1000, lit('High')).\
        when(col('amount')>=50, lit('Medium')).\
            otherwise('Low')).\
                otherwise('Invalid'))


df.show()

### 🐍 Day 2 — Python Practice #1

##### Topic: Python fundamentals for Data Engineering

We'll practice:

- Variables & data types
- if / elif / else
- Lists
- Dictionaries
- for loops
- Functions
- Basic data-processing logic

###### 🎯 Challenge 1 — Order Validation

`orders = [
    {"order_id": "O101", "amount": 1500, "quantity": 2},
    {"order_id": "O102", "amount": -500, "quantity": 1},
    {"order_id": "O103", "amount": 800, "quantity": None},
    {"order_id": "O104", "amount": 1200, "quantity": 3},
]`

Your task:

1. Loop through the orders.

2. If amount < 0, mark the order as "INVALID".

3. If quantity is None, replace it with 1.

4. Calculate:

order_value = amount × quantity

5. Print:

- O101 → VALID → 3000
- O102 → INVALID
- O103 → VALID → 800
- O104 → VALID → 3600

In [0]:
orders = [
    {"order_id": "O101", "amount": 1500, "quantity": 2},
    {"order_id": "O102", "amount": -500, "quantity": 1},
    {"order_id": "O103", "amount": 800, "quantity": None},
    {"order_id": "O104", "amount": 1200, "quantity": 3},
]
for order in orders:
    order_id=order['order_id']
    amount=order['amount']
    quantity=order['quantity']

    if amount<0:
        print(f"{order_id}-> Invalid")
    else:
        if quantity is None:
            quantity=1
        order_value=quantity*amount
        print(f"{order_id}-> valid {order_value}") 



#### 🟢 Q2 — Customer Spending

You receive:

`orders = [
    {"order_id": "O101", "customer_id": "C001", "amount": 1500},
    {"order_id": "O102", "customer_id": "C002", "amount": 800},
    {"order_id": "O103", "customer_id": "C001", "amount": 1200},
    {"order_id": "O104", "customer_id": "C003", "amount": 500},
    {"order_id": "O105", "customer_id": "C002", "amount": 700},
]`
Requirement

Calculate the total amount spent by each customer.

Expected:

C001 -> 2700
C002 -> 1500
C003 -> 500
Restrictions

Use only:

for loop
dictionary
if
.get()

No Pandas, PySpark, Counter, or other libraries.

In [0]:
orders = [
    {"order_id": "O101", "customer_id": "C001", "amount": 1500},
    {"order_id": "O102", "customer_id": "C002", "amount": 800},
    {"order_id": "O103", "customer_id": "C001", "amount": 1200},
    {"order_id": "O104", "customer_id": "C003", "amount": 500},
    {"order_id": "O105", "customer_id": "C002", "amount": 700},
]

total_amount={}

for order in orders:
    customer_id=order['customer_id']
    amount=order['amount']

    total_amount[customer_id]=total_amount.get(customer_id,0)+amount
print(total_amount)
    

Find Duplicate Orders

Now a different scenario.

You receive order IDs from a source system:

`order_ids = [
    "O101",
    "O102",
    "O103",
    "O101",
    "O104",
    "O102",
    "O105",
    "O106",
    "O103"
]`

Requirement

Find which order IDs are duplicated.

In [0]:
order_ids = [
    "O101",
    "O102",
    "O103",
    "O101",
    "O104",
    "O102",
    "O105",
    "O106",
    "O103"
]

seen=set()
duplicates=set()

for order_id in order_ids:
    if order_id in seen:
        duplicates.add(order_id)
    else:
        seen.add(order_id)

print(list(duplicates))

Remove Duplicates

Now slightly different.

Given:

`customer_ids = [
    "C001",
    "C002",
    "C001",
    "C003",
    "C002",
    "C004",
    "C003"
]`
Requirement

Create a new list containing each customer only once.

Expected:

C001
C002
C003
C004
Restrictions

Use:

for loop
set
if

Don't use `set(customer_ids)` directly.

In [0]:
customer_ids = [
    "C001",
    "C002",
    "C001",
    "C003",
    "C002",
    "C004",
    "C003"
]

seen=set()
duplicates=set()
for customer_id in customer_ids:
    if customer_id in seen:
        duplicates.add(customer_id)
    else:
        seen.add(customer_id)

print(seen)

Python Q5/20 — Highest Transaction

Given:

transactions = `[450, 1200, 800, 2500, 950, 1800]`

Find the highest transaction amount.

Restrictions:

Use a for loop
Use if
Don't use max()

Expected output:

In [0]:
transactions = [450, 1200, 800, 2500, 950, 1800]
x=0
for t in transactions:
    if t>x:
        x=t 
print(x)


Python Q6/20 — Count Orders by Status

Now let's work with a real data-engineering type of problem.

`orders = [
    {"order_id": "O101", "status": "Completed"},
    {"order_id": "O102", "status": "Pending"},
    {"order_id": "O103", "status": "Completed"},
    {"order_id": "O104", "status": "Cancelled"},
    {"order_id": "O105", "status": "Completed"},
    {"order_id": "O106", "status": "Pending"},
]`

Find how many orders belong to each status.

Restrictions: for loop + dictionary + .get().
Don't use Counter.

In [0]:
orders = [
    {"order_id": "O101", "status": "Completed"},
    {"order_id": "O102", "status": "Pending"},
    {"order_id": "O103", "status": "Completed"},
    {"order_id": "O104", "status": "Cancelled"},
    {"order_id": "O105", "status": "Completed"},
    {"order_id": "O106", "status": "Pending"},
]
order_status={}
for order in orders:
    order_id=order['order_id']
    status=order['status']
    order_status[status]=order_status.get(status,0)+1
print(order_status)


Python Q7/20 — Clean Customer Names

Now let's change the type of problem.

`customers = [
    "  Shivam  ",
    "RAHUL",
    " priya ",
    "AMIT  ",
    "  neha"
]`

Clean the names so the output becomes:

Shivam
Rahul
Priya
Amit
Neha
Requirements

Use:

for loop
.strip()
.lower() or .upper()
.capitalize()

Create a new list called cleaned_customers.

In [0]:
customers = [
    "  Shivam  ",
    "RAHUL",
    " priya ",
    "AMIT  ",
    "  neha"
]

cleaned_customer=[]

for customer in customers:
    cleaned_customer.append(customer.strip().capitalize())

print(cleaned_customer)

Python Q8/20 — Handle Missing Values

Now let's test None handling.

`customers = [
    {"customer_id": "C001", "name": "Shivam", "city": "Delhi"},
    {"customer_id": "C002", "name": None, "city": "Mumbai"},
    {"customer_id": "C003", "name": "Rahul", "city": None},
    {"customer_id": "C004", "name": None, "city": None},
]`
Requirement

For every customer:

If name is None → replace with "Unknown"
If city is None → replace with "Unknown"

Expected:

Use:

for
if
None

Don't use Pandas/PySpark.

In [0]:
customers = [
    {"customer_id": "C001", "name": "Shivam", "city": "Delhi"},
    {"customer_id": "C002", "name": None, "city": "Mumbai"},
    {"customer_id": "C003", "name": "Rahul", "city": None},
    {"customer_id": "C004", "name": None, "city": None},
]

for customer in customers:
    if customer['name']==None:
        customer['name']='Unknown'
    if customer['city']==None:
        customer['city']='Unknown'

    print(customer)

    



Q9/20 — Filter Expensive Orders

Given:

`orders = [
    {"order_id": "O101", "amount": 1500},
    {"order_id": "O102", "amount": 450},
    {"order_id": "O103", "amount": 2200},
    {"order_id": "O104", "amount": 800},
    {"order_id": "O105", "amount": 3000},
]`

Find all orders where amount > 1000.

Expected:

O101
O103
O105

Create a new list called expensive_orders.

Use:

for
if
append()

Don't use filter() or Pandas/PySpark.

Your turn.

In [0]:
orders = [
    {"order_id": "O101", "amount": 1500},
    {"order_id": "O102", "amount": 450},
    {"order_id": "O103", "amount": 2200},
    {"order_id": "O104", "amount": 800},
    {"order_id": "O105", "amount": 3000},
]

filter_order=[]
for order in orders:
    order_id=order['order_id']
    amount=order['amount']
    if amount>1000:
        filter_order.append(order_id)

print(filter_order)

🐍 Python Challenge 10/20 — Average Transaction

Given:

`transactions = [500, 1200, 800, 1500, 1000]`

Calculate the average transaction amount.

Rules

Use:

for loop
variables
arithmetic

❌ Don't use sum()
❌ Don't use len()

Expected output:

In [0]:
transactions = [500, 1200, 800, 1500, 1000]
total_amount=0
no_transaction=0
for transaction in transactions:
    total_amount=total_amount+transaction
    no_transaction=no_transaction+1

average=total_amount/no_transaction
print(average)


🐍 Challenge 11/20 — Group Products

Now let's make it more Data Engineering-like.

`orders = [
    {"order_id": "O101", "product": "Laptop", "amount": 1500},
    {"order_id": "O102", "product": "Mobile", "amount": 800},
    {"order_id": "O103", "product": "Laptop", "amount": 1200},
    {"order_id": "O104", "product": "Tablet", "amount": 500},
    {"order_id": "O105", "product": "Mobile", "amount": 700},
]`
Requirement

Calculate total sales amount for each product.

Rules

Use:

- for
- dictionary
- .get()
- addition
- 
- ❌ No Pandas
- ❌ No PySpark
- ❌ No Counter

In [0]:
orders = [
    {"order_id": "O101", "product": "Laptop", "amount": 1500},
    {"order_id": "O102", "product": "Mobile", "amount": 800},
    {"order_id": "O103", "product": "Laptop", "amount": 1200},
    {"order_id": "O104", "product": "Tablet", "amount": 500},
    {"order_id": "O105", "product": "Mobile", "amount": 700},
]
total_amount={}

for order in orders:
    product=order['product']
    amount=order['amount']
    total_amount[product]=total_amount.get(product,0)+amount

print(total_amount)




🐍 Challenge 12/20 — Customers With Multiple Orders

Now let's increase the difficulty slightly.

Given:

`orders = [
    {"order_id": "O101", "customer_id": "C001"},
    {"order_id": "O102", "customer_id": "C002"},
    {"order_id": "O103", "customer_id": "C001"},
    {"order_id": "O104", "customer_id": "C003"},
    {"order_id": "O105", "customer_id": "C002"},
    {"order_id": "O106", "customer_id": "C001"},
]`
Requirement

Find customers who have more than one order.

Expected:

C001
C002
Rules

- Use:
- 
- for
- dictionary
- .get()
- if
- 
- ❌ No Counter
- ❌ No Pandas/PySpark


In [0]:
orders = [
    {"order_id": "O101", "customer_id": "C001"},
    {"order_id": "O102", "customer_id": "C002"},
    {"order_id": "O103", "customer_id": "C001"},
    {"order_id": "O104", "customer_id": "C003"},
    {"order_id": "O105", "customer_id": "C002"},
    {"order_id": "O106", "customer_id": "C001"},
]
duplicate=set()
seen=set()

for order in orders:
    order_id=order['order_id']
    customer_id=order['customer_id']
    if customer_id in seen:
        duplicate.add(customer_id)
    else:
        seen.add(customer_id)
print(duplicate)
            
    
    
   


🐍 Challenge 13/20 — Validate Records

Now we'll move into functions, which are very important for real ETL code.

Given:
`
orders = [
    {"order_id": "O101", "amount": 1500},
    {"order_id": "O102", "amount": -500},
    {"order_id": "O103", "amount": 800},
    {"order_id": "O104", "amount": None},
]`

Create a function:

`validate_order(order)`

It should return:

O101 → Valid
O102 → Invalid
O103 → Valid
O104 → Invalid
Rules

An order is Valid when:

- amount is not None
- amount > 0
- 
- Otherwise → Invalid
- 
- Use:
- 
- def
- if/else
- dictionary access
- return



In [0]:
def validate_order(order):
    order_id=order['order_id']
    amount=order['amount']
    if amount is None or amount<=0:
        return 'Invalid'
    else:
        return 'Valid'
    




In [0]:
orders = [
    {"order_id": "O101", "amount": 1500},
    {"order_id": "O102", "amount": -500},
    {"order_id": "O103", "amount": 800},
    {"order_id": "O104", "amount": None},
]
for order in orders:
    result=validate_order(order)
    print(order['order_id'],result)

🐍 Challenge 14/20 — Error Handling

Now we're going to learn try/except, which is important when processing dirty source data.

Suppose the source gives us:

`amounts = ["1500", "800", "invalid", "1200", "abc"]`

We want to convert each value to an integer.

Expected:

- 1500
- 800
- Invalid amount
- 1200
- Invalid amount

Rules

Use:

- for
- try
- except
- int()

Don't let "invalid" or "abc" crash the entire program.

In [0]:
amounts = ["1500", "800", "invalid", "1200", "abc"]
for amount in amounts:
    try:
        converted_amount=int(amount)
        print(converted_amount)
    except:
        print('Invalid amount')
            



🐍 Challenge 15/20 — Process JSON-like Data

Now we'll work with a structure you'll see constantly in Python ETL:

customers = [
    {"customer_id": "C001", "name": "Shivam", "city": "Delhi"},
    {"customer_id": "C002", "name": "Rahul", "city": "Mumbai"},
    {"customer_id": "C003", "name": "Priya", "city": "Delhi"},
]
Requirement

Create a new dictionary that stores customer name → city.

Expected:

 {
-     "Shivam": "Delhi",
-     "Rahul": "Mumbai",
-     "Priya": "Delhi"
 }

Rules

Use:

- for
- dictionary
- dictionary assignment

❌ No Pandas
❌ No PySpark

This one should be straightforward based on what you've already learned. 💪

In [0]:
customers = [
    {"customer_id": "C001", "name": "Shivam", "city": "Delhi"},
    {"customer_id": "C002", "name": "Rahul", "city": "Mumbai"},
    {"customer_id": "C003", "name": "Priya", "city": "Delhi"},
]
customer_city={}
for customer in customers:
    name=customer['name']
    city=customer['city']
    customer_city[name]=city
print(customer_city)

🐍 Challenge 16/20 — Transform Records

Now we're moving toward real ETL logic.

Given:

`orders = [
    {"order_id": "O101", "amount": 1500, "quantity": 2},
    {"order_id": "O102", "amount": 800, "quantity": 1},
    {"order_id": "O103", "amount": 1200, "quantity": 3},
]`

Create a function:

transform_order(order)

It should calculate:

total_value = amount × quantity

and return a new dictionary containing:

`{
    "order_id": "O101",
    "total_value": 3000
}`

For all records, expected output:

- O101 → 3000
- O102 → 800
- O103 → 3600
Rules

Use:

def
dictionary access
multiplication
return
for loop

Important: Don't modify the original order dictionary. Return a new dictionary.

In [0]:
def transform_order(orders):
    order_id=orders['order_id']
    amount=orders['amount']
    quantity=orders['quantity']
    order_amount={}


    total_amount=quantity*amount
    order_amount[order_id]=total_amount
    return order_amount

orders = [
    {"order_id": "O101", "amount": 1500, "quantity": 2},
    {"order_id": "O102", "amount": 800, "quantity": 1},
    {"order_id": "O103", "amount": 1200, "quantity": 3},
]

for order in orders:
    print(transform_order(order))


🐍 Challenge 17/20 — Date Filtering

Now let's introduce dates, which are extremely important in Data Engineering.

from datetime import datetime

`orders = [
    {"order_id": "O101", "order_date": "2026-08-01", "amount": 1500},
    {"order_id": "O102", "order_date": "2026-08-05", "amount": 800},
    {"order_id": "O103", "order_date": "2026-08-10", "amount": 1200},
    {"order_id": "O104", "order_date": "2026-08-15", "amount": 900},
]`
Requirement

Find orders placed on or after August 10, 2026.

Expected:

O103
O104
Rules

Use:

datetime.strptime()
for
if
date comparison

💡 Hint:

Convert the string:

"2026-08-10"

into a real date using:

datetime.strptime(date_string, "%Y-%m-%d")

Then compare it with the target date.

In [0]:
from datetime import datetime

orders = [
    {"order_id": "O101", "order_date": "2026-08-01", "amount": 1500},
    {"order_id": "O102", "order_date": "2026-08-05", "amount": 800},
    {"order_id": "O103", "order_date": "2026-08-10", "amount": 1200},
    {"order_id": "O104", "order_date": "2026-08-15", "amount": 900},
]
for order in orders:
    order_id=order['order_id']
    order_date=order['order_date']
    order_date=datetime.strptime(order_date,"%Y-%m-%d")
    target_date = datetime.strptime("2026-08-10", "%Y-%m-%d")
    if order_date>=target_date:
        print(order_id)


🐍 Challenge 18/20 — Reusable ETL Function

Now we're getting closer to real ETL code.

Given:

`orders = [
    {"order_id": "O101", "amount": 1500, "quantity": 2},
    {"order_id": "O102", "amount": -500, "quantity": 1},
    {"order_id": "O103", "amount": 800, "quantity": 3},
    {"order_id": "O104", "amount": 1200, "quantity": 2},
]`

Create a function:

process_order(order)
Rules

The function should:

Check whether amount > 0.
If invalid → return:
{"order_id": "O102", "status": "Invalid"}
If valid → calculate:
total_value = amount × quantity

and return:

{"order_id": "O101", "status": "Valid", "total_value": 3000}

Then process all orders using a for loop.

Expected:

O101 → Valid → 3000
O102 → Invalid
O103 → Valid → 2400
O104 → Valid → 2400
Restrictions

Use:

def
if/else
dictionary
return
for
multiplication

This is your first mini ETL function combining several concepts you've learned.

In [0]:

def process_order(order):
    order_id=order['order_id']
    amount=order['amount']
    quantity=order['quantity']
    if amount<=0:
        return {
    "order_id": order_id,
    "status": "Invalid"
    }
    else:
        total_value=quantity*amount
        return {
            "order_id":order_id,
            "status":"Valid",
            "total_value":total_value
        }
        

orders = [
    {"order_id": "O101", "amount": 1500, "quantity": 2},
    {"order_id": "O102", "amount": -500, "quantity": 1},
    {"order_id": "O103", "amount": 800, "quantity": 3},
    {"order_id": "O104", "amount": 1200, "quantity": 2},
]
for order in orders:
    print(process_order(order))



Perfect. Q19 — Debugging a Broken ETL Function 🔥

This time don't write new code from scratch. Your job is to find the mistakes and fix them.

Scenario

You're processing orders before loading them into a data warehouse.

`def clean_order(order):

    order_id = order["order_id"]
    amount = order["amount"]
    quantity = order["quantity"]

    if amount <= 0:
        return {
            "order_id": order_id,
            "status": "Invalid"
        }

    total_value = amount * quantity

    return {
        "order_id": order_id,
        "status": "Valid",
        "total_value": total_value
    }`


`orders = [
    {"order_id": "O101", "amount": "1500", "quantity": 2},
    {"order_id": "O102", "amount": "-500", "quantity": 1},
    {"order_id": "O103", "amount": "800", "quantity": 3},
    {"order_id": "O104", "amount": "invalid", "quantity": 2},
]
`
for order in orders:
    result = clean_order(order)
    print(result)
Your task

This code has a runtime/data-type problem.

Identify why it will fail.
Fix the function so that:
"1500" becomes numeric 1500
"-500" becomes numeric -500
"invalid" should not crash the pipeline

invalid records should return:

{"order_id": "O104", "status": "Invalid"}
Use try/except for the conversion.
Don't change the input orders.

Expected result:

O101 → Valid → 3000
O102 → Invalid
O103 → Valid → 2400
O104 → Invalid

Your turn: rewrite only the clean_order() function first.

In [0]:
def clean_order(order):

    order_id = order["order_id"]
    amount = order["amount"]
    quantity = order["quantity"]
    try:
        converted_amount=int(amount)
        if converted_amount <= 0:
            return {
                "order_id": order_id,
                
                "status": "Invalid"
            }
        else:

            total_value = converted_amount * quantity

            return {
            "order_id": order_id,
            "status": "Valid",
            "total_value": total_value
            }

    except:
        return {
            "order_id" :'Invalid'
        }
    

    


orders = [
    {"order_id": "O101", "amount": "1500", "quantity": 2},
    {"order_id": "O102", "amount": "-500", "quantity": 1},
    {"order_id": "O103", "amount": "800", "quantity": 3},
    {"order_id": "O104", "amount": "invalid", "quantity": 2},
]

for order in orders:
    result = clean_order(order)
    print(result)

Absolutely. Q20 — Final Python Data Engineering Challenge 🔥

This combines almost everything you've practiced so far: functions, loops, dictionaries, validation, aggregation, and error handling.

Scenario

You receive raw order data from an API:

orders = [
    {"order_id": "O101", "customer_id": "C001", "amount": "1500", "quantity": 2},
    {"order_id": "O102", "customer_id": "C002", "amount": "800", "quantity": 1},
    {"order_id": "O103", "customer_id": "C001", "amount": "-500", "quantity": 1},
    {"order_id": "O104", "customer_id": "C003", "amount": "1200", "quantity": 3},
    {"order_id": "O105", "customer_id": "C002", "amount": "invalid", "quantity": 2},
    {"order_id": "O106", "customer_id": "C001", "amount": "700", "quantity": 2},
]
Your task

Build a small Python ETL pipeline.

Step 1 — Create a function

Create:

def process_order(order):

It should:

Convert amount from string to integer.
If conversion fails → mark order as "Invalid".
If amount <= 0 → mark order as "Invalid".
Otherwise calculate:
total_value = amount × quantity

Return a dictionary like:

{
    "order_id": "O101",
    "customer_id": "C001",
    "status": "Valid",
    "total_value": 3000
}

For invalid records:

{
    "order_id": "O103",
    "customer_id": "C001",
    "status": "Invalid"
}
Step 2 — Process all orders

Use a for loop and create a list:

processed_orders = []

Add every processed record to this list.

Step 3 — Calculate customer spending

Only consider Valid orders.

Create a dictionary:

customer_spending = {}

Calculate the total total_value for each customer.

Expected:

C001 → 4400
C002 → 800
C003 → 3600
Step 4 — Find the highest-spending customer

Using your dictionary, find:

Customer: C001
Spending: 4400
Final expected output

Something approximately like:

Processed Orders:
O101 → Valid → 3000
O102 → Valid → 800
O103 → Invalid
O104 → Valid → 3600
O105 → Invalid
O106 → Valid → 1400

Customer Spending:
C001 → 4400
C002 → 800
C003 → 3600

Highest Spending Customer:
C001 → 4400

Don't use Pandas or PySpark. Do this using only Python basics we've practiced.

In [0]:


def process_order(order):
    order_id=order['order_id']
    customer_id=order['customer_id']
    amount=order['amount']
    quantity=order['quantity']

    try:
        converted_amount=int(amount)
        if converted_amount<=0:
            return {
                "order_id":order_id,
                "customer_id":customer_id,
                "status":"Invalid"
            }

        else:
            total_value=converted_amount*quantity
            return {
                "order_id":order_id,
                "customer_id":customer_id,
                "status":"Valid",
                "total_value":total_value

            }

    except:
        return{
            "order_id":order_id,
            "customer_id":customer_id,
            "status":"Invalid"
        }

orders = [
    {"order_id": "O101", "customer_id": "C001", "amount": "1500", "quantity": 2},
    {"order_id": "O102", "customer_id": "C002", "amount": "800", "quantity": 1},
    {"order_id": "O103", "customer_id": "C001", "amount": "-500", "quantity": 1},
    {"order_id": "O104", "customer_id": "C003", "amount": "1200", "quantity": 3},
    {"order_id": "O105", "customer_id": "C002", "amount": "invalid", "quantity": 2},
    {"order_id": "O106", "customer_id": "C001", "amount": "700", "quantity": 2},
]

processed_orders=[]
customer_spending = {}

for order in orders:
    result=process_order(order)
    processed_orders.append(result)
    customer_id=result['customer_id']
    total_value=result['total_value']
    customer_spending[customer_id] = customer_spending.get(customer_id, 0) + total_value
    
   

